 # BBC News IR Assignment — Part A: Text Processing & Part B: Vocabulary and Indexing

 **Dataset:** BBC News raw text dataset (2,225 documents; business, entertainment,

 politics, sport, tech)


 **Scope of this notebook:** Part A (Text Processing) and Part B (Vocabulary and

 Indexing) only. Boolean retrieval, tolerant retrieval, and evaluation (Parts C, D, E)

 are implemented by other team members in separate notebooks and are **out of scope here**.



 ## Reproducibility and imports

In [1]:
import platform
import sys

print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")



Python version: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]
Platform: Windows-11-10.0.26100-SP0


In [2]:
import importlib

_PACKAGES = ("nltk", "pandas", "matplotlib")
for _pkg in _PACKAGES:
    _mod = importlib.import_module(_pkg)
    print(f"{_pkg}: {getattr(_mod, '__version__', 'unknown')}")



nltk: 3.10.3
pandas: 2.3.3
matplotlib: 3.11.1


In [3]:
import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
print(f"Random seed fixed at {RANDOM_SEED}")



Random seed fixed at 42


In [4]:
from pathlib import Path

# Resolve the repository root robustly whether this notebook is opened from the
# project root or from inside the `notebooks/` folder. We do this by walking up
# from the current working directory until we find a directory that contains
# both `notebooks` and `datasets`.
def find_repo_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing both
    `notebooks` and `datasets` subfolders is found.
    """
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from "
        f"{start}. Expected to find sibling 'notebooks' and 'datasets' folders."
    )


REPO_ROOT = find_repo_root(Path.cwd())
print(f"Repository root: {REPO_ROOT}")



Repository root: C:\Users\sanjaytharan.tamilse\OneDrive - Autoliv\Engineer_Sanjaytharan\Programming\Python\Sem 2\IR_Assignment_1


In [5]:
# Create output directories if missing. We do not change the global working
# directory anywhere in this notebook; all paths are built from REPO_ROOT.
OUTPUT_FIGURES_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_TABLES_DIR = REPO_ROOT / "outputs" / "tables"
OUTPUT_INDEXES_DIR = REPO_ROOT / "outputs" / "indexes"

for _dir in (OUTPUT_FIGURES_DIR, OUTPUT_TABLES_DIR, OUTPUT_INDEXES_DIR):
    _dir.mkdir(parents=True, exist_ok=True)
    print(f"Ready: {_dir}")



Ready: C:\Users\sanjaytharan.tamilse\OneDrive - Autoliv\Engineer_Sanjaytharan\Programming\Python\Sem 2\IR_Assignment_1\outputs\figures
Ready: C:\Users\sanjaytharan.tamilse\OneDrive - Autoliv\Engineer_Sanjaytharan\Programming\Python\Sem 2\IR_Assignment_1\outputs\tables
Ready: C:\Users\sanjaytharan.tamilse\OneDrive - Autoliv\Engineer_Sanjaytharan\Programming\Python\Sem 2\IR_Assignment_1\outputs\indexes


 ## Configuration

In [6]:
DATASET_ROOT = REPO_ROOT / "datasets" / "bbc-fulltext" / "bbc"
EXPECTED_CATEGORIES = (
    "business",
    "entertainment",
    "politics",
    "sport",
    "tech",
)
EXPECTED_DOCUMENT_COUNT = 2225
ENCODING_CANDIDATES = ("utf-8", "latin-1")

print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"EXPECTED_CATEGORIES: {EXPECTED_CATEGORIES}")
print(f"EXPECTED_DOCUMENT_COUNT: {EXPECTED_DOCUMENT_COUNT}")



DATASET_ROOT: C:\Users\sanjaytharan.tamilse\OneDrive - Autoliv\Engineer_Sanjaytharan\Programming\Python\Sem 2\IR_Assignment_1\datasets\bbc-fulltext\bbc
EXPECTED_CATEGORIES: ('business', 'entertainment', 'politics', 'sport', 'tech')
EXPECTED_DOCUMENT_COUNT: 2225


 ## Dataset validation and loading



 The helper functions below are generic setup utilities (file discovery, encoding

 fallback, validation) and are provided complete. They are not part of the assessed

 text-processing or indexing logic in Parts A and B.



 Document ID policy: `"{category}/{file_stem}"`, e.g. `"business/001"`. This is

 deterministic (based only on the file's path) and unique because BBC filenames are

 unique within each category.



In [7]:
def validate_dataset_root(dataset_root: Path, expected_categories) -> None:
    """Confirm the dataset root and all expected category folders exist."""
    if not dataset_root.is_dir():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")
    missing = [c for c in expected_categories if not (dataset_root / c).is_dir()]
    if missing:
        raise FileNotFoundError(f"Missing expected category folders: {missing}")


validate_dataset_root(DATASET_ROOT, EXPECTED_CATEGORIES)
print("Dataset root and category folders validated.")



Dataset root and category folders validated.


In [8]:
def read_text_with_fallback(path: Path, encodings) -> str:
    """Read a text file trying each encoding in order, raising if all fail."""
    last_error = None
    for encoding in encodings:
        try:
            return path.read_text(encoding=encoding)
        except (UnicodeDecodeError, LookupError) as exc:
            last_error = exc
    raise ValueError(f"Could not decode {path} with {encodings}: {last_error}")


def discover_documents(dataset_root: Path, expected_categories):
    """Deterministically find all BBC article files.

    Returns a sorted list of (doc_id, category, path) tuples. README.TXT files
    and any non-'.txt' files are excluded. Category label text itself is never
    treated as a token later on.
    """
    records = []
    for category in sorted(expected_categories):
        category_dir = dataset_root / category
        for file_path in sorted(category_dir.glob("*.txt")):
            doc_id = f"{category}/{file_path.stem}"
            records.append((doc_id, category, file_path))
    records.sort(key=lambda r: r[0])
    return records


_raw_records = discover_documents(DATASET_ROOT, EXPECTED_CATEGORIES)
_doc_ids = [r[0] for r in _raw_records]
_duplicate_ids = {d for d in _doc_ids if _doc_ids.count(d) > 1}
if _duplicate_ids:
    raise ValueError(f"Duplicate document IDs detected: {_duplicate_ids}")

print(f"Discovered {len(_raw_records)} candidate document files.")



Discovered 2225 candidate document files.


In [9]:
documents = []
_empty_file_count = 0
_unreadable_file_count = 0

for doc_id, category, file_path in _raw_records:
    try:
        text = read_text_with_fallback(file_path, ENCODING_CANDIDATES)
    except ValueError as exc:
        _unreadable_file_count += 1
        print(f"Unreadable file skipped: {file_path} ({exc})")
        continue
    if not text.strip():
        _empty_file_count += 1
    relative_path = file_path.relative_to(REPO_ROOT)
    documents.append(
        {
            "doc_id": doc_id,
            "category": category,
            "path": str(relative_path).replace("\\", "/"),
            "text": text,
        }
    )

documents.sort(key=lambda d: d["doc_id"])

_category_counts = {}
for _doc in documents:
    _category_counts[_doc["category"]] = _category_counts.get(_doc["category"], 0) + 1

print(f"Loaded documents: {len(documents)}")
print(f"Category counts: {_category_counts}")
print(f"Empty files: {_empty_file_count}")
print(f"Unreadable files: {_unreadable_file_count}")

if len(documents) != EXPECTED_DOCUMENT_COUNT:
    print(
        f"WARNING: expected {EXPECTED_DOCUMENT_COUNT} documents, "
        f"found {len(documents)}. Investigate before proceeding."
    )


Loaded documents: 2225
Category counts: {'business': 510, 'entertainment': 386, 'politics': 417, 'sport': 511, 'tech': 401}
Empty files: 0
Unreadable files: 0


 ## Text Preprocessing

 The aim of preprocessing is to convert each raw BBC News article into a clean

 and consistent sequence of terms that can later be used to build a vocabulary

 and an inverted index. I keep every intermediate version instead of only the

 final output. This is important because the assignment requires a comparison

 of how each operation changes the corpus.



 The stages used in this implementation are:



 1. **Tokenization**: split raw text into word tokens.

 2. **Case normalization**: convert tokens to lowercase.

 3. **Stop-word removal**: remove very common English function words.

 4. **Porter stemming**: reduce words using rule-based suffix stripping.

 5. **WordNet lemmatization**: reduce words to dictionary base forms using POS tags.



 Stemming and lemmatization are treated as two alternative final branches. Both

 branches receive the same lowercase, stop-word-filtered input, which makes the

 comparison fair.

In [10]:
import re
from collections import Counter

import nltk
import pandas as pd
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer



 ### NLTK resources

 NLTK stores some language resources separately from the Python package. The

 helper below first checks whether a resource is already available. A download

 is attempted only when the resource is missing. This makes repeated notebook

 runs faster and avoids unnecessary downloads.

In [11]:
def ensure_nltk_resource(resource_path: str, download_name: str) -> None:
    """Check for an NLTK resource and download it only when it is missing."""
    try:
        nltk.data.find(resource_path)
    except LookupError:
        print(f"NLTK resource '{download_name}' is missing. Attempting download...")
        if not nltk.download(download_name, quiet=True):
            raise RuntimeError(
                f"Unable to download '{download_name}'. Please install this NLTK "
                "resource before running the preprocessing section."
            )


ensure_nltk_resource("corpora/stopwords", "stopwords")
ensure_nltk_resource("corpora/wordnet", "wordnet")

# NLTK versions use one of the following names for the English POS tagger.
try:
    nltk.data.find("taggers/averaged_perceptron_tagger_eng")
except LookupError:
    try:
        ensure_nltk_resource(
            "taggers/averaged_perceptron_tagger_eng",
            "averaged_perceptron_tagger_eng",
        )
    except RuntimeError:
        ensure_nltk_resource(
            "taggers/averaged_perceptron_tagger",
            "averaged_perceptron_tagger",
        )

STOP_WORDS = set(stopwords.words("english"))
PORTER_STEMMER = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()

print(f"English stop words loaded: {len(STOP_WORDS):,}")



NLTK resource 'wordnet' is missing. Attempting download...
English stop words loaded: 198


 ### Tokenization policy


 A token is defined here as a sequence of alphabetic characters with an optional

 internal apostrophe. For example, `BBC's` and `don't` are retained as tokens,

 while punctuation marks, numbers, and standalone symbols are excluded.

 Hyphenated expressions are separated into individual words. This explicit rule

 gives deterministic results and does not require NLTK's Punkt sentence data.



 This policy is suitable for a word-based Boolean retrieval system. A limitation

 is that numeric expressions such as years and prices are not indexed. This is a

 deliberate design choice for the current assignment and should be mentioned

 when interpreting retrieval results.

In [12]:
TOKEN_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def tokenize(text: str) -> list[str]:
    """Extract word-like tokens while preserving their original case."""
    return TOKEN_PATTERN.findall(text)


def normalize_case(tokens: list[str]) -> list[str]:
    """Convert every token to lowercase so case variants share one term."""
    return [token.lower() for token in tokens]


def remove_stop_words(tokens: list[str]) -> list[str]:
    """Remove tokens found in NLTK's English stop-word list."""
    return [token for token in tokens if token not in STOP_WORDS]



 ### Porter stemming


 The Porter stemmer applies a sequence of suffix-removal rules. It does not look

 up a word in a dictionary, so the result may be a non-word. For example,

 `government` may become `govern`, while `studies` may become `studi`.



 In information retrieval, this can improve recall by combining morphological

 variants under one stem. However, aggressive stemming may also combine terms

 that should remain different. This is one reason the assignment compares

 stemming with lemmatization instead of assuming one method is always better.

In [13]:
def stem_tokens(tokens: list[str]) -> list[str]:
    """Apply Porter's stemming algorithm to a sequence of tokens."""
    return [PORTER_STEMMER.stem(token) for token in tokens]



 ### POS-aware lemmatization

 Lemmatization aims to return a valid dictionary base form, called a lemma.

 WordNet lemmatization works better when the grammatical role of each token is

 supplied. Therefore, Penn Treebank POS tags are mapped to the noun, verb,

 adjective, and adverb categories expected by WordNet.



 During the first run, the output showed the undesirable transformation

 `us -> u`. In news articles, lowercase `us` can represent the pronoun "us" or

 the country abbreviation "US" after case normalization. Since both meanings

 become identical after lowercasing, this implementation protects `us` and

 leaves it unchanged. This safeguard avoids creating the meaningless index term

 `u`, although it cannot recover the original country/pronoun distinction.

In [14]:
def penn_to_wordnet(tag: str) -> str:
    """Map a Penn Treebank POS tag to the corresponding WordNet POS value."""
    first_letter = tag[0].upper()
    return {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "R": wordnet.ADV,
        "V": wordnet.VERB,
    }.get(first_letter, wordnet.NOUN)


def lemmatize_tokens(tokens: list[str]) -> list[str]:
    """Apply POS-aware WordNet lemmatization with a safeguard for 'us'."""
    tagged_tokens = nltk.pos_tag(tokens)
    lemmas = []

    for token, tag in tagged_tokens:
        # Preserve 'us' because WordNet can incorrectly reduce it to 'u'.
        if token == "us":
            lemma = "us"
        else:
            lemma = LEMMATIZER.lemmatize(token, penn_to_wordnet(tag))
        lemmas.append(lemma)

    return lemmas



 ### Complete preprocessing function



 The following function performs the stages in a fixed order and returns every

 intermediate representation. Stemming and lemmatization are parallel outputs,

 not consecutive operations. Applying both one after the other would make it

 difficult to determine which method caused a vocabulary change.

In [15]:
def preprocess_document(text: str) -> dict[str, list[str]]:
    """Create all preprocessing representations for one raw document."""
    tokenized = tokenize(text)
    normalized = normalize_case(tokenized)
    no_stopwords = remove_stop_words(normalized)

    return {
        "tokenized": tokenized,
        "normalized": normalized,
        "no_stopwords": no_stopwords,
        "stemmed": stem_tokens(no_stopwords),
        "lemmatized": lemmatize_tokens(no_stopwords),
    }


for document in documents:
    document.update(preprocess_document(document["text"]))

print(f"Preprocessed {len(documents):,} documents.")
print(
    "Representations stored for each document: tokenized, normalized, "
    "no_stopwords, stemmed, lemmatized"
)



Preprocessed 2,225 documents.
Representations stored for each document: tokenized, normalized, no_stopwords, stemmed, lemmatized


 ### Inspect one document at every stage

 A fixed document is displayed so the example remains reproducible. Only the

 first 30 tokens are printed because complete articles would make the notebook

 difficult to read. This output provides evidence that each transformation is

 being applied in the intended order.

In [16]:
STAGE_LABELS = {
    "tokenized": "Tokenization, original case",
    "normalized": "After case normalization",
    "no_stopwords": "After stop-word removal",
    "stemmed": "After Porter stemming",
    "lemmatized": "After WordNet lemmatization",
}

example_document = documents[0]
print(f"Example document: {example_document['doc_id']}")
print(f"Category: {example_document['category']}")

for stage, label in STAGE_LABELS.items():
    print(f"\n{label}:")
    print(example_document[stage][:30])



Example document: business/001
Category: business

Tokenization, original case:
['Ad', 'sales', 'boost', 'Time', 'Warner', 'profit', 'Quarterly', 'profits', 'at', 'US', 'media', 'giant', 'TimeWarner', 'jumped', 'to', 'bn', 'm', 'for', 'the', 'three', 'months', 'to', 'December', 'from', 'm', 'year', 'earlier', 'The', 'firm', 'which']

After case normalization:
['ad', 'sales', 'boost', 'time', 'warner', 'profit', 'quarterly', 'profits', 'at', 'us', 'media', 'giant', 'timewarner', 'jumped', 'to', 'bn', 'm', 'for', 'the', 'three', 'months', 'to', 'december', 'from', 'm', 'year', 'earlier', 'the', 'firm', 'which']

After stop-word removal:
['ad', 'sales', 'boost', 'time', 'warner', 'profit', 'quarterly', 'profits', 'us', 'media', 'giant', 'timewarner', 'jumped', 'bn', 'three', 'months', 'december', 'year', 'earlier', 'firm', 'one', 'biggest', 'investors', 'google', 'benefited', 'sales', 'high', 'speed', 'internet', 'connections']

After Porter stemming:
['ad', 'sale', 'boost', 'time', 'warn

 ### Corpus-level preprocessing statistics

 Two measurements are especially important for this assignment:

 - **Total tokens** count all term occurrences in the corpus.

 - **Vocabulary size** counts distinct terms.

 Case normalization should normally reduce vocabulary size without changing the

 number of tokens. Stop-word removal should reduce token count considerably.

 Stemming and lemmatization should preserve token count while reducing vocabulary

 by merging related surface forms.

In [17]:
def corpus_statistics(document_collection: list[dict], field: str) -> dict:
    """Calculate token, vocabulary, and document-length statistics for a stage."""
    frequencies = Counter(
        token
        for document in document_collection
        for token in document[field]
    )
    document_lengths = [len(document[field]) for document in document_collection]

    return {
        "stage": field,
        "total_tokens": sum(frequencies.values()),
        "vocabulary_size": len(frequencies),
        "average_document_length": (
            sum(document_lengths) / len(document_lengths)
            if document_lengths else 0.0
        ),
        "minimum_document_length": min(document_lengths, default=0),
        "maximum_document_length": max(document_lengths, default=0),
        "most_frequent_terms": frequencies.most_common(10),
    }


stage_statistics = [
    corpus_statistics(documents, stage)
    for stage in STAGE_LABELS
]

baseline_tokens = stage_statistics[0]["total_tokens"]
baseline_vocabulary = stage_statistics[0]["vocabulary_size"]

statistics_table = pd.DataFrame(stage_statistics)
statistics_table["stage"] = statistics_table["stage"].map(STAGE_LABELS)

# Both percentages use the tokenized stage as the common baseline. This makes
# the accumulated effect of the complete pipeline easy to compare.
statistics_table["token_reduction_percent"] = (
    100 * (baseline_tokens - statistics_table["total_tokens"]) / baseline_tokens
)
statistics_table["vocabulary_reduction_percent"] = (
    100
    * (baseline_vocabulary - statistics_table["vocabulary_size"])
    / baseline_vocabulary
)

statistics_display = statistics_table.drop(columns=["most_frequent_terms"]).copy()
for column in [
    "average_document_length",
    "token_reduction_percent",
    "vocabulary_reduction_percent",
]:
    statistics_display[column] = statistics_display[column].round(2)

print("\nPreprocessing comparison:")
print(statistics_display.to_string(index=False))

statistics_csv_path = OUTPUT_TABLES_DIR / "preprocessing_statistics.csv"
statistics_table.to_csv(statistics_csv_path, index=False)
print(f"\nSaved statistics to: {statistics_csv_path}")




Preprocessing comparison:
                      stage  total_tokens  vocabulary_size  average_document_length  minimum_document_length  maximum_document_length  token_reduction_percent  vocabulary_reduction_percent
Tokenization, original case        847950            33765                   381.10                       89                     4417                     0.00                          0.00
   After case normalization        847950            29503                   381.10                       89                     4417                     0.00                         12.62
    After stop-word removal        486779            29327                   218.78                       48                     2205                    42.59                         13.14
      After Porter stemming        486779            20578                   218.78                       48                     2205                    42.59                         39.06
After WordNet lemmatization 

 ### Most frequent terms at each stage



 Frequent-term lists give a quick qualitative check. Before stop-word removal,

 common function words such as `the` and `of` should dominate. Afterwards,

 content-bearing terms should become more visible. Stemmed forms may not be valid

 dictionary words, while lemmas should usually remain readable.

In [18]:
for record in stage_statistics:
    print(f"\n{STAGE_LABELS[record['stage']]}")
    print(record["most_frequent_terms"])




Tokenization, original case
[('the', 44610), ('to', 24997), ('of', 19904), ('and', 18038), ('a', 17250), ('in', 16703), ('for', 8728), ('is', 8546), ('The', 8022), ('that', 7803)]

After case normalization
[('the', 52636), ('to', 25113), ('of', 20008), ('and', 18612), ('a', 18342), ('in', 17734), ('for', 8945), ('is', 8555), ('that', 8055), ('on', 7624)]

After stop-word removal
[('said', 7255), ('mr', 3005), ('would', 2581), ('also', 2156), ('year', 2088), ('new', 1978), ('people', 1971), ('us', 1956), ('one', 1870), ('could', 1510)]

After Porter stemming
[('said', 7255), ('year', 3091), ('mr', 3046), ('would', 2581), ('also', 2156), ('new', 1978), ('peopl', 1972), ('us', 1956), ('one', 1916), ('time', 1667)]

After WordNet lemmatization
[('say', 8843), ('year', 3091), ('mr', 3023), ('would', 2581), ('make', 2251), ('also', 2156), ('new', 1996), ('us', 1981), ('people', 1972), ('one', 1916)]


 ### Representative stemming and lemmatization examples



 The controlled examples below show the conceptual difference between the two

 methods. A second table is then produced from terms actually found in the BBC

 corpus. The corpus table is more useful as assignment evidence because it shows

 that the differences occurred in the chosen dataset.

In [19]:
representative_words = [
    "studies",
    "studying",
    "relational",
    "connections",
    "better",
    "running",
    "agreed",
    "policies",
    "universities",
    "generously",
    "us",
]

controlled_comparison = pd.DataFrame({
    "original": representative_words,
    "porter_stem": stem_tokens(representative_words),
    "wordnet_lemma": lemmatize_tokens(representative_words),
})
controlled_comparison["stem_differs_from_lemma"] = (
    controlled_comparison["porter_stem"]
    != controlled_comparison["wordnet_lemma"]
)

print("Controlled examples:")
print(controlled_comparison.to_string(index=False))



Controlled examples:
    original porter_stem wordnet_lemma  stem_differs_from_lemma
     studies       studi         study                     True
    studying       studi         study                     True
  relational       relat    relational                     True
 connections     connect    connection                     True
      better      better          well                     True
     running         run           run                    False
      agreed        agre        agreed                     True
    policies      polici        policy                     True
universities     univers  universities                     True
  generously       gener    generously                     True
          us          us            us                    False


In [20]:
# Compare the aligned outputs produced for every corpus token. The Counter keeps
# the most common differences so the final report can use representative rather
# than rare or accidental examples.
corpus_differences = Counter()

for document in documents:
    for original, stem, lemma in zip(
        document["no_stopwords"],
        document["stemmed"],
        document["lemmatized"],
    ):
        if stem != lemma:
            corpus_differences[(original, stem, lemma)] += 1

corpus_difference_table = pd.DataFrame(
    [
        {
            "original": original,
            "porter_stem": stem,
            "wordnet_lemma": lemma,
            "corpus_count": count,
        }
        for (original, stem, lemma), count
        in corpus_differences.most_common(20)
    ]
)

print("\nMost frequent stemming and lemmatization differences in the corpus:")
print(corpus_difference_table.to_string(index=False))

comparison_csv_path = OUTPUT_TABLES_DIR / "stemming_vs_lemmatization_examples.csv"
corpus_difference_table.to_csv(comparison_csv_path, index=False)
print(f"\nSaved comparison examples to: {comparison_csv_path}")




Most frequent stemming and lemmatization differences in the corpus:
  original porter_stem wordnet_lemma  corpus_count
      said        said           say          7255
    people       peopl        people          1971
government      govern    government          1030
      made        made          make           862
      told        told          tell           862
      many        mani          many           830
  election       elect      election           662
     added          ad           add           654
   company     compani       company           619
     since        sinc         since           607
technology   technolog    technology           561
    mobile       mobil        mobile           542
     party       parti         party           542
  minister      minist      minister           519
   however       howev       however           514
   already     alreadi       already           473
   service      servic       service           450
   economy   

## Objective

The objective of preprocessing was to transform the raw BBC News articles into a consistent and standardized representation suitable for vocabulary construction and inverted indexing. The preprocessing pipeline consisted of five major stages:

1. Tokenization
2. Case normalization
3. Stop-word removal
4. Porter stemming
5. WordNet lemmatization

To evaluate the effect of each preprocessing operation, corpus-level statistics such as total token count, vocabulary size, and average document length were recorded after every stage.

---

## Preprocessing Statistics

| Stage | Total Tokens | Vocabulary Size | Average Document Length |
|---------|---------:|---------:|---------:|
| Tokenization (Original Case) | 847,950 | 33,765 | 381.10 |
| Case Normalization | 847,950 | 29,503 | 381.10 |
| Stop-word Removal | 486,779 | 29,327 | 218.78 |
| Porter Stemming | 486,779 | 20,578 | 218.78 |
| WordNet Lemmatization | 486,779 | 24,703 | 218.78 |

---

## Analysis of Individual Preprocessing Stages

### Tokenization

The BBC corpus contained **847,950 tokens** distributed across **2,225 news articles**, with an average document length of approximately **381 tokens per document**.

At this stage, punctuation was removed and the text was split into individual word tokens while preserving the original casing. The vocabulary consisted of **33,765 unique terms**, representing the complete set of distinct words before any normalization.

The most frequent tokens were:

```text
the, to, of, and, a, in, for, is, The, that
```

These results indicate the presence of many common English function words and case variations, which motivates later preprocessing steps.

---

### Case Normalization

Case normalization converted all tokens to lowercase. The total number of tokens remained unchanged at **847,950**, while the vocabulary size decreased from **33,765** to **29,503** terms.

This corresponds to a vocabulary reduction of approximately **12.62%**.

The reduction occurred because words that differed only by capitalization were merged into a single representation.

Examples include:

```text
The → the
BBC → bbc
News → news
```

This operation improves retrieval effectiveness because users generally do not distinguish between uppercase and lowercase search terms.

---

### Stop-word Removal

After removing English stop words, the total token count decreased significantly from **847,950** to **486,779**, representing a reduction of approximately **42.59%**.

The average document length also decreased from:

```text
381.10 tokens → 218.78 tokens
```

Common stop words such as:

```text
the
of
to
and
in
for
```

were removed because they appear extremely frequently but carry little semantic meaning for retrieval.

Interestingly, the vocabulary size only decreased slightly:

```text
29,503 → 29,327
```

This demonstrates that stop words contribute heavily to overall token frequency but represent only a small proportion of the unique vocabulary.

The most frequent remaining terms became:

```text
said, mr, would, also, year, new, people, us, one, could
```

which are much more informative than the original function words.

---

## Porter Stemming Analysis

Porter stemming reduced the vocabulary from **29,327** terms to **20,578** terms while keeping the total number of tokens unchanged.

This represents a vocabulary reduction of approximately **29.83%** relative to the stop-word-filtered vocabulary.

Porter stemming works by applying rule-based suffix removal to merge related word forms. Examples from the BBC corpus include:

| Original Word | Stem |
|---------------|------|
| people | peopl |
| government | govern |
| company | compani |
| technology | technolog |
| minister | minist |
| economy | economi |

### Advantages of Stemming

- Produces the smallest vocabulary.
- Reduces storage requirements for indexing.
- Improves recall by merging morphological variants.

### Limitations of Stemming

Porter stems are not always valid English words.

Examples include:

```text
people → peopl
company → compani
minister → minist
```

A particularly notable example found in the corpus is:

```text
added → ad
```

This represents a case of overstemming because the generated stem may become ambiguous and potentially match unrelated words.

---

## WordNet Lemmatization Analysis

WordNet lemmatization reduced the vocabulary to **24,703** terms, which is larger than the stemmed vocabulary but significantly smaller than the original vocabulary.

Unlike stemming, lemmatization attempts to produce a valid dictionary base form by using lexical knowledge and part-of-speech information.

Examples from the corpus include:

| Original Word | Lemma |
|--------------|-------|
| said | say |
| made | make |
| told | tell |
| companies | company |
| services | service |

### Advantages of Lemmatization

- Produces meaningful dictionary words.
- Handles irregular word forms correctly.
- Preserves semantic meaning more effectively.

### Limitations of Lemmatization

- Less aggressive than stemming.
- Produces a larger vocabulary.
- Requires part-of-speech tagging, making it computationally more expensive.

---

## Stemming versus Lemmatization

To compare the two approaches, representative examples were extracted from both controlled test cases and the actual BBC corpus.

### Controlled Examples

| Original | Porter Stem | WordNet Lemma |
|-----------|-------------|---------------|
| studies | studi | study |
| connections | connect | connection |
| policies | polici | policy |
| agreed | agre | agreed |
| better | better | well |

### Most Frequent Differences Observed in the BBC Corpus

| Original | Porter Stem | WordNet Lemma | Frequency |
|-----------|-------------|---------------|---------:|
| said | said | say | 7,255 |
| people | peopl | people | 1,971 |
| government | govern | government | 1,030 |
| made | made | make | 862 |
| told | told | tell | 862 |
| added | ad | add | 654 |
| company | compani | company | 619 |
| technology | technolog | technology | 561 |

These examples demonstrate that stemming focuses on reducing words through rule-based suffix removal, whereas lemmatization attempts to recover the actual base form of each word.

As a result, stemming achieves greater vocabulary compression, while lemmatization preserves readability and linguistic correctness.

---

## Frequent-Term Analysis

### After Stop-word Removal

```text
said
mr
would
also
year
new
people
us
one
could
```

These terms indicate that stop-word removal successfully eliminated common grammatical words and exposed content-bearing vocabulary.

### After Porter Stemming

```text
said
year
mr
would
also
new
peopl
us
one
time
```

Several non-dictionary stems appear, such as `peopl`, illustrating the aggressive nature of stemming.

### After WordNet Lemmatization

```text
say
year
mr
would
make
also
new
us
people
one
```

The resulting vocabulary remains human-readable while still reducing vocabulary size.

---

## Key Findings

The preprocessing experiments produced several important observations:

- Case normalization reduced vocabulary size by **12.62%** without changing the number of tokens.
- Stop-word removal reduced the total number of tokens by **42.59%**.
- Porter stemming achieved the greatest vocabulary reduction, decreasing the vocabulary to **20,578 terms**.
- WordNet lemmatization produced a larger vocabulary (**24,703 terms**) but maintained meaningful dictionary forms.
- Both stemming and lemmatization preserved token counts while reducing vocabulary through term conflation.
- Stemming improved vocabulary compression, whereas lemmatization produced cleaner and more interpretable terms.

---

The preprocessing pipeline successfully transformed the BBC News corpus into standardized term representations suitable for information retrieval tasks.

The results demonstrate that each preprocessing stage contributes differently to corpus normalization. Case normalization merged capitalization variants, stop-word removal eliminated high-frequency non-content terms, stemming aggressively reduced vocabulary size, and lemmatization preserved linguistically meaningful word forms.

For subsequent vocabulary construction and indexing experiments, both stemmed and lemmatized representations were retained. This allows the impact of stemming and lemmatization on retrieval effectiveness to be analyzed in later stages of the Information Retrieval system.